# No-Comma Activation Steering

vLLM-Hook is an extensible framework that aims to allow selective access to model internals during inference.
In this notebook, we demonstrate a Phi-3 activation-steering vector that nudges generations away from comma usage.

The steering vector was derived from real Phi-3 hidden states using no-comma instruction-following examples. The notebook compares the same prompts with and without the hook enabled and counts commas in each output.


### Installation

If running this from a new environment, set `RUN_INSTALL = True` in the cell below to install `vllm_hook_plugins` and the repository requirements.
The default keeps installation disabled so an already-prepared local or cluster environment can execute the notebook without reinstalling packages.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
PKG_DIR = REPO_ROOT / "vllm_hook_plugins"
REQ_FILE = REPO_ROOT / "requirement.txt"
RUN_INSTALL = False

print("Notebook dir:", NOTEBOOK_DIR)
print("Repo root   :", REPO_ROOT)
print("Package dir :", PKG_DIR)
print("Req file    :", REQ_FILE)

if RUN_INSTALL:
    if REQ_FILE.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)], check=True)
    else:
        print("Warning: requirement.txt not found at", REQ_FILE)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PKG_DIR)], check=True)
else:
    print("Install step skipped. Set RUN_INSTALL=True if dependencies are not installed.")

plugin_src_dir = str(PKG_DIR.resolve())
if plugin_src_dir not in sys.path:
    sys.path.insert(0, plugin_src_dir)


### Importing the Hook-Enabled LLM

The plugin provides its own LLM wrapper that behaves like `vllm.LLM` but adds support for hooks and instrumentation.


In [ ]:
from vllm_hook_plugins import HookLLM


### Environment & Multiprocessing Setup

In [ ]:
import os
import multiprocessing as mp
from pathlib import Path

import torch
from vllm import SamplingParams

mp.set_start_method("spawn", force=True)
os.environ["VLLM_USE_V1"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

try:
    REPO_ROOT
except NameError:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO_ROOT)
print("Working directory:", Path.cwd())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU name :", torch.cuda.get_device_name(0))


### Initialize `HookLLM`

The no-comma steering config uses `add_vector` on Phi-3 layer 15 with a coefficient of 40 and applies the vector to all tokens.


In [4]:
import json

cache_dir = "./cache/"
model = "microsoft/Phi-3-mini-4k-instruct"
json_path = Path("model_configs/activation_steer/Phi-3-mini-4k-instruct-no-comma.json")

with open(json_path, "r") as f:
    config = json.load(f)

vector_path = Path(config["steering"]["vector_path"])
print(json.dumps(config, indent=2))
print("Config file    :", json_path)
print("Steering vector:", vector_path, "exists=", vector_path.exists())


{
  "steering": {
    "method": "add_vector",
    "coefficient": 40,
    "optimal_layer": 15,
    "vector_path": "steering_vectors/phi3_no_comma.pt",
    "apply_at_all_positions": true,
    "apply_to_all_tokens": true
  }
}
Config file    : model_configs/activation_steer/Phi-3-mini-4k-instruct-no-comma.json
Steering vector: steering_vectors/phi3_no_comma.pt exists= True


In [5]:
llm = HookLLM(
    model=model,
    worker_name="steer_hook_act",
    config_file=str(json_path),
    download_dir=cache_dir,
    gpu_memory_utilization=0.6,
    max_model_len=2048,
    trust_remote_code=True,
    dtype="auto",
    enforce_eager=True,
    enable_prefix_caching=True,
    enable_hook=True,
    tensor_parallel_size=1,
)


INFO 06-04 10:39:28 [utils.py:278] non-default args: {'trust_remote_code': True, 'download_dir': './cache/', 'max_model_len': 2048, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.6, 'disable_log_stats': True, 'enforce_eager': True, 'worker_extension_cls': 'vllm_hook_plugins.workers.steer_activation_worker.SteerHookActWorker', 'model': 'microsoft/Phi-3-mini-4k-instruct'}


WARNING 06-04 10:39:28 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_USE_V1


INFO 06-04 10:39:30 [model.py:617] Resolved architecture: Phi3ForCausalLM


INFO 06-04 10:39:30 [model.py:1752] Using max model len 2048


INFO 06-04 10:39:30 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.


...

(EngineCore pid=3508703) INFO 06-04 10:40:13 [default_loader.py:397] Loading weights took 12.68 seconds


(EngineCore pid=3508703) INFO 06-04 10:40:14 [gpu_model_runner.py:5132] Model loading took 7.12 GiB memory and 13.651288 seconds


(EngineCore pid=3508703) INFO 06-04 10:40:16 [gpu_worker.py:466] Available KV cache memory: 39.8 GiB
(EngineCore pid=3508703) INFO 06-04 10:40:16 [kv_cache_utils.py:1733] GPU KV cache size: 107,829 tokens
(EngineCore pid=3508703) INFO 06-04 10:40:16 [kv_cache_utils.py:1734] Maximum concurrency for 2,048 tokens per request: 52.65x
(EngineCore pid=3508703) INFO 06-04 10:40:16 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=3508703) INFO 06-04 10:40:16 [core.py:309] init engine (profile, create kv cache, warmup model) took 2.24 s


(EngineCore pid=3508703) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(EngineCore pid=3508703) INFO 06-04 10:40:18 [vllm.py:977] Asynchronous scheduling is enabled.
(EngineCore pid=3508703) WARNING 06-04 10:40:18 [vllm.py:1033] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=3508703) WARNING 06-04 10:40:18 [vllm.py:1058] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=3508703) INFO 06-04 10:40:18 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=3508703) INFO 06-04 10:40:18 [vllm.py:1234] Cudagraph is disabled under eager mode
(EngineCore pid=3508703) INFO 06-04 10:40:18 [compilation.py:312] Enabled custom fusions: norm_quant, act_quant


### Test Cases

Each prompt asks for a useful response without commas. The comparison below runs the same prompts with and without the activation-steering hook.


In [6]:
def comma_count(text: str) -> int:
    return text.count(",")


prompts = [
    (
        "I am planning a trip to Japan and I would like thee to write an itinerary "
        "for my journey in a Shakespearean style. You are not allowed to use any "
        "commas in your response."
    ),
    (
        "Write a short product profile for a travel mug that keeps coffee hot for "
        "a full workday. Do not use any commas in your response."
    ),
]

prompts


['I am planning a trip to Japan and I would like thee to write an itinerary for my journey in a Shakespearean style. You are not allowed to use any commas in your response.',
 'Write a short product profile for a travel mug that keeps coffee hot for a full workday. Do not use any commas in your response.']

### Sampling Parameters

Token `32007` is Phi-specific and is included as an additional stop token, matching the existing Phi-3 activation-steering demo.


In [7]:
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=220,
    stop_token_ids=[llm.tokenizer.eos_token_id, 32007],
)


### Generate With and Without Steering

In [8]:
examples = [
    llm.tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=False,
    )
    for prompt in prompts
]

steered_outputs = llm.generate(examples, sampling_params)
llm.llm_engine.reset_prefix_cache()
baseline_outputs = llm.generate(examples, sampling_params, use_hook=False)
llm.llm_engine.reset_prefix_cache()


Rendering prompts:   0%|          | 0/2 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 2/2 [00:00<00:00, 27.18it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=3508703) Installed 32 steering hooks on layers: ['model.layers.0', 'model.layers.1', 'model.layers.2', 'model.layers.3', 'model.layers.4', 'model.layers.5', 'model.layers.6', 'model.layers.7', 'model.layers.8', 'model.layers.9', 'model.layers.10', 'model.layers.11', 'model.layers.12', 'model.layers.13', 'model.layers.14', 'model.layers.15', 'model.layers.16', 'model.layers.17', 'model.layers.18', 'model.layers.19', 'model.layers.20', 'model.layers.21', 'model.layers.22', 'model.layers.23', 'model.layers.24', 'model.layers.25', 'model.layers.26', 'model.layers.27', 'model.layers.28', 'model.layers.29', 'model.layers.30', 'model.layers.31']
(EngineCore pid=3508703) Hooks installed successfully
(EngineCore pid=3508703) WARNING 06-04 10:40:18 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts:  50%|█████     | 1/2 [00:00<00:00,  1.09it/s, est. speed input: 36.13 toks/s, output: 40.51 toks/s]

...

Processed prompts: 100%|██████████| 2/2 [00:03<00:00,  1.57s/it, est. speed input: 23.34 toks/s, output: 108.09 toks/s]

Processed prompts: 100%|██████████| 2/2 [00:03<00:00,  1.57s/it, est. speed input: 23.34 toks/s, output: 108.09 toks/s]

Processed prompts: 100%|██████████| 2/2 [00:03<00:00,  1.63s/it, est. speed input: 23.34 toks/s, output: 108.09 toks/s]

(EngineCore pid=3508703) INFO 06-04 10:40:23 [block_pool.py:482] Successfully reset prefix cache


True

### Results

In [9]:
rows = []

for prompt, steered, baseline in zip(prompts, steered_outputs, baseline_outputs):
    steered_text = steered.outputs[0].text
    baseline_text = baseline.outputs[0].text
    rows.append(
        {
            "prompt": prompt,
            "steered_commas": comma_count(steered_text),
            "baseline_commas": comma_count(baseline_text),
        }
    )

    print("=" * 80)
    print("[Prompt]")
    print(prompt)

    print("\n[With no-comma activation steering]")
    print(steered_text)
    print(f"Commas: {comma_count(steered_text)}")

    print("\n[Without activation steering]")
    print(baseline_text)
    print(f"Commas: {comma_count(baseline_text)}")

print("=" * 80)
print("Summary:")
for row in rows:
    print(
        f"steered_commas={row['steered_commas']} | "
        f"baseline_commas={row['baseline_commas']} | "
        f"prompt={row['prompt'][:70]}..."
    )


[Prompt]
I am planning a trip to Japan and I would like thee to write an itinerary for my journey in a Shakespearean style. You are not allowed to use any commas in your response.

[With no-comma activation steering]
 Hark! Thou art embarking on a grand adventure. Tokyo first, then Kyoto. From there, Osaka. Nara, Hiroshima, and finally, Tokyo. Thou must visit Mount Fuji. Thy journey ends.
Commas: 5

[Without activation steering]
 Hark! Thou art embarking on a grand voyage to the Land of the Rising Sun, Japan. Pray, attend to this itinerary, penned in the style of the Bard himself.


Day the First: Arrival in Tokyo, the heart of Nippon. Thou shalt alight at the bustling Shibuya Station, where the throngs of people doth mirror the stars in the night sky. Thy lodging shall be a humble inn, where the warmth of the hearth and the comfort of the bed shall welcome thee.


Day the Second: Venture forth to the Imperial Palace, a fortress of history and culture. Thou shalt marvel at the gardens,